# Notebook 2 — Clasificador SVC → MongoDB
### Investing AI · iDeSo · UNMSM

**Objetivo:** Leer datos de `precios_ohlcv` en MongoDB, entrenar un clasificador SVC  
con GridSearchCV para predecir señales BUY/SELL, y guardar predicciones y métricas  
en las colecciones `predicciones` y `metricas_modelos`.

**Prerequisito:** Haber ejecutado el Notebook 1 (colección `precios_ohlcv` poblada).  
**Salida:** Colecciones `predicciones` y `metricas_modelos` listas para la API.

> Inspirado en la investigación doctoral del Prof. Cancho-Rodríguez sobre SVM con optimización.

## Paso 1 — Instalación de librerías

In [ ]:
# Instalación de librerías necesarias
!pip install 'pymongo[srv]' scikit-learn --quiet
print('✓ Librerías instaladas')

## Paso 2 — Conexión a MongoDB Atlas

In [ ]:
# Conectar a MongoDB Atlas usando el Secret MONGO_URI de Colab
# (Secrets: ícono de llave en el panel izquierdo de Colab)
from google.colab import userdata
from pymongo import MongoClient
import pandas as pd
import numpy as np
from datetime import datetime

MONGO_URI = userdata.get('MONGO_URI')
client    = MongoClient(MONGO_URI)
db        = client['spbi']

# Verificar conexión
client.admin.command('ping')
print('✓ Conectado a MongoDB Atlas')
print(f'  Base de datos: spbi')
print(f'  Colecciones disponibles: {db.list_collection_names()}')

## Paso 3 — Carga de datos desde MongoDB

In [ ]:
# Función para cargar precios OHLCV de un ticker desde MongoDB
def cargar_datos(ticker):
    """
    Lee los documentos de precios_ohlcv para un ticker
    y los devuelve como DataFrame ordenado por fecha.
    """
    docs = list(db['precios_ohlcv'].find({'ticker': ticker}).sort('fecha', 1))
    if not docs:
        return pd.DataFrame()
    df = pd.DataFrame(docs)
    df['fecha'] = pd.to_datetime(df['fecha'])
    df = df.sort_values('fecha').reset_index(drop=True)
    return df

# Verificación rápida con BVN
df_test = cargar_datos('BVN')
print(f'✓ BVN: {len(df_test)} registros cargados')
print(df_test[['fecha','close','sma_20','rsi_14']].tail(3).to_string(index=False))

## Paso 4 — Ingeniería de features y variable objetivo

In [ ]:
# Features para tickers con precio variable (FSM, ABX.TO, BVN, BHP)
FEATURES_PRECIO = ['close','sma_20','sma_50','ema_12','ema_26','rsi_14','retorno']

# Features especiales para VOLCABC1.LM (precio constante → usar volumen)
FEATURES_VOLUMEN = ['vol_change_pct','vol_relative','vol_rsi','obv_slope','money_flow_rel']

def preparar_features_precio(df):
    """
    Feature engineering estándar basado en precio e indicadores técnicos.
    Target: 1 (BUY) si el cierre del día siguiente es mayor, 0 (SELL) si baja.
    Partición temporal 80/20 — sin data leakage.
    """
    df = df.copy()
    df['retorno'] = df['close'].pct_change()
    # Target: ¿sube mañana?
    df['target'] = (df['close'].shift(-1) > df['close']).astype(int)
    df = df.dropna(subset=FEATURES_PRECIO + ['target'])

    X = df[FEATURES_PRECIO].values
    y = df['target'].values
    fechas = df['fecha'].values
    precios = df['close'].values
    return X, y, fechas, precios

def preparar_features_volumen(df):
    """
    Feature engineering basado en VOLUMEN para VOLCABC1.LM.
    Usado porque el precio de ese ticker es casi constante y los
    indicadores técnicos estándar no aportan señal estadística útil.
    """
    df = df.copy()
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0)

    # Cambio porcentual diario de volumen
    df['vol_change_pct'] = df['volume'].pct_change().fillna(0).clip(-5, 5)
    # Volumen relativo respecto a su media de 20 días
    df['vol_sma20']      = df['volume'].rolling(20).mean()
    df['vol_relative']   = (df['volume'] / df['vol_sma20'].replace(0, np.nan)).fillna(1).clip(0, 10)
    # OBV (On-Balance Volume)
    direction    = np.sign(df['close'].diff().fillna(0))
    df['obv']    = (direction * df['volume']).cumsum()
    df['obv_sma20']  = df['obv'].rolling(20).mean()
    df['obv_slope']  = df['obv'].diff(5).fillna(0)
    # RSI aplicado al volumen
    delta_vol = df['volume'].diff()
    gain_vol  = delta_vol.clip(lower=0).rolling(14).mean()
    loss_vol  = (-delta_vol.clip(upper=0)).rolling(14).mean()
    rs_vol    = gain_vol / loss_vol.replace(0, np.nan)
    df['vol_rsi'] = (100 - 100 / (1 + rs_vol)).fillna(50)
    # Money flow relativo
    df['money_flow']       = df['close'] * df['volume']
    df['mf_sma10']         = df['money_flow'].rolling(10).mean()
    df['money_flow_rel']   = (df['money_flow'] / df['mf_sma10'].replace(0, np.nan)).fillna(1)

    # Target: señal basada en OBV
    df['target'] = df.apply(
        lambda r: 1 if (r['obv'] > r['obv_sma20'] and r['vol_relative'] > 1.3)
                  else (0 if (r['obv'] <= r['obv_sma20'] and r['vol_relative'] > 1.3)
                        else -1), axis=1
    )
    df = df[df['target'] != -1].dropna(subset=FEATURES_VOLUMEN)

    X = df[FEATURES_VOLUMEN].values
    y = df['target'].values
    fechas  = df['fecha'].values
    precios = df['close'].values
    return X, y, fechas, precios

print('✓ Funciones de feature engineering definidas')

## Paso 5 — Entrenamiento SVC con GridSearchCV

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight

def entrenar_svc(X, y):
    """
    Entrena SVC con GridSearchCV sobre un conjunto temporal 80/20.
    - class_weight='balanced': corrige desbalance de clases (evita F1=0.000)
    - cv=5 folds con scoring f1_weighted
    - Pipeline: StandardScaler + SVC (evita data leakage en validación cruzada)
    """
    n     = len(X)
    corte = int(n * 0.80)
    X_train, X_test = X[:corte], X[corte:]
    y_train, y_test = y[:corte], y[corte:]

    # Pipeline: escalado + clasificador
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('svc',    SVC(probability=True, class_weight='balanced'))
    ])

    param_grid = {
        'svc__kernel': ['linear', 'rbf'],
        'svc__C'     : [0.1, 1, 10],
        'svc__gamma' : ['scale', 'auto']
    }

    grid = GridSearchCV(
        pipeline, param_grid,
        cv=5, scoring='f1_weighted',
        n_jobs=-1, refit=True
    )
    grid.fit(X_train, y_train)

    y_pred = grid.predict(X_test)

    metricas = {
        'accuracy' : round(float(accuracy_score(y_test,  y_pred)),                                4),
        'precision': round(float(precision_score(y_test, y_pred, average='weighted', zero_division=0)), 4),
        'recall'   : round(float(recall_score(y_test,    y_pred, average='weighted', zero_division=0)), 4),
        'f1'       : round(float(f1_score(y_test,        y_pred, average='weighted', zero_division=0)), 4),
        'mejor_kernel': grid.best_params_['svc__kernel'],
        'mejor_C'     : grid.best_params_['svc__C'],
        'mejor_gamma' : str(grid.best_params_['svc__gamma']),
        'n_train'  : int(len(X_train)),
        'n_test'   : int(len(X_test))
    }

    cm = confusion_matrix(y_test, y_pred).tolist()

    return grid, metricas, cm, X_test, y_test

print('✓ Función entrenar_svc definida')
print('  Mejoras respecto a la versión base:')
print('  - class_weight=balanced (evita F1=0.000 por desbalance)')
print('  - Pipeline StandardScaler+SVC (sin leakage en CV)')
print('  - cv=5 folds, scoring=f1_weighted')

## Paso 6 — Pipeline completo por ticker

In [ ]:
def procesar_ticker(ticker):
    """
    Pipeline completo para un ticker:
    1. Carga datos de MongoDB
    2. Selecciona feature engineering (precio o volumen según ticker)
    3. Entrena SVC con GridSearchCV
    4. Genera historial de señales sobre todos los datos
    5. Guarda predicción actual + métricas + historial en MongoDB
    """
    print(f'\n  [{ticker}] Procesando...')

    df = cargar_datos(ticker)
    if len(df) < 100:
        print(f'  [{ticker}] ✗ Datos insuficientes ({len(df)} registros) — se omite')
        return False

    # VOLCABC1.LM usa features de volumen (precio casi constante)
    if ticker == 'VOLCABC1.LM':
        X, y, fechas, precios = preparar_features_volumen(df)
        tipo_features = 'volumen'
    else:
        X, y, fechas, precios = preparar_features_precio(df)
        tipo_features = 'precio'

    if len(X) < 50:
        print(f'  [{ticker}] ✗ Muestras insuficientes tras limpieza — se omite')
        return False

    print(f'  [{ticker}] Features: {tipo_features} | Muestras: {len(X)}')

    grid, metricas, cm, X_test, y_test = entrenar_svc(X, y)

    # ── Señal actual (última observación disponible) ───────────
    ultima_X = X[-1:].reshape(1, -1)
    pred_actual = int(grid.predict(ultima_X)[0])
    prob_actual = float(grid.predict_proba(ultima_X)[0].max())
    senal_actual = 'BUY' if pred_actual == 1 else 'SELL'

    # ── Historial de señales (todos los datos) ─────────────────
    preds_all  = grid.predict(X)
    probas_all = grid.predict_proba(X)
    buy_idx    = list(grid.classes_).index(1) if 1 in grid.classes_ else 0

    historico_senales = [
        {
            'fecha'      : str(fechas[i])[:10],
            'precio'     : round(float(precios[i]), 4),
            'prediccion' : 'BUY' if preds_all[i] == 1 else 'SELL',
            'probabilidad': round(float(probas_all[i][buy_idx]), 4)
        }
        for i in range(len(preds_all))
    ]

    # ── Guardar predicción actual en MongoDB ───────────────────
    db['predicciones'].delete_many({'ticker': ticker, 'modelo': 'SVC'})
    db['predicciones'].insert_one({
        'ticker'           : ticker,
        'modelo'           : 'SVC',
        'senal'            : senal_actual,
        'confianza'        : round(prob_actual, 4),
        'fecha_prediccion' : datetime.now().strftime('%Y-%m-%d'),
        'historico_senales': historico_senales,
        'tipo_features'    : tipo_features,
        'created_at'       : datetime.now()
    })

    # ── Guardar métricas en MongoDB ────────────────────────────
    db['metricas_modelos'].delete_many({'ticker': ticker, 'modelo': 'SVC'})
    db['metricas_modelos'].insert_one({
        'ticker'             : ticker,
        'modelo'             : 'SVC',
        **metricas,
        'matriz_confusion'   : cm,
        'tipo_features'      : tipo_features,
        'fecha_entrenamiento': datetime.now()
    })

    a  = metricas['accuracy']
    f1 = metricas['f1']
    k  = metricas['mejor_kernel']
    c  = metricas['mejor_C']
    print(f'  [{ticker}] ✓ senal={senal_actual} ({prob_actual:.0%}) | acc={a:.0%} | f1={f1:.0%} | kernel={k}, C={c}')
    return True

print('✓ Función procesar_ticker definida')

## Paso 7 — Ejecutar pipeline para los 5 tickers

In [ ]:
TICKERS = ['FSM', 'VOLCABC1.LM', 'ABX.TO', 'BVN', 'BHP']

print('=' * 55)
print('  CLASIFICADOR SVC — ENTRENAMIENTO PARA 5 TICKERS')
print('=' * 55)

exitosos = 0
for ticker in TICKERS:
    try:
        ok = procesar_ticker(ticker)
        if ok:
            exitosos += 1
    except Exception as e:
        print(f'  [{ticker}] ✗ Error inesperado: {e}')

print()
print('=' * 55)
print(f'  ✅ Pipeline completo: {exitosos}/{len(TICKERS)} tickers procesados')
print('=' * 55)

## Paso 8 — Verificación en MongoDB

In [ ]:
print('PREDICCIONES SVC en MongoDB')
print('-' * 45)
for doc in db['predicciones'].find({'modelo': 'SVC'}, {'_id':0,'historico_senales':0,'created_at':0}):
    t  = doc['ticker']
    s  = doc['senal']
    c  = doc['confianza']
    tf = doc.get('tipo_features','precio')
    print(f'  {t:<15} {s:<5} ({c:.0%}) | features: {tf}')

print()
print('MÉTRICAS SVC en MongoDB')
print('-' * 45)
for doc in db['metricas_modelos'].find({'modelo': 'SVC'}, {'_id':0,'matriz_confusion':0,'fecha_entrenamiento':0}):
    t  = doc['ticker']
    a  = doc['accuracy']
    f1 = doc['f1']
    p  = doc['precision']
    r  = doc['recall']
    k  = doc.get('mejor_kernel','?')
    c  = doc.get('mejor_C','?')
    print(f'  {t:<15} acc={a:.0%} | f1={f1:.0%} | prec={p:.0%} | rec={r:.0%} | {k}/C={c}')

print()
print('✅ Colecciones predicciones y metricas_modelos listas para la API')

## Resultado

Las colecciones `predicciones` y `metricas_modelos` contienen resultados reales del SVC:
- **5 tickers** procesados (VOLCABC1.LM con features de volumen)
- **class_weight='balanced'** aplicado → F1 correcto incluso en clases desbalanceadas
- **historico_senales** almacenado → el frontend puede graficar precio + señales BUY/SELL
- **Sin data leakage** → partición temporal 80/20 estricta

**La API (Notebook 3)** leerá estas colecciones y las servirá al frontend vía HTTP.

**Pipeline completo:** `precios_ohlcv` → SVC → `predicciones` + `metricas_modelos` ✓